# Tabular foundation models on a GPU

A guided run on Kaggle's free T4. Three things happen here that could not
happen on a laptop CPU:

1. **You see what a GPU actually does**, measured rather than described —
   including the timing mistake almost everyone makes the first time.
2. **The 5,000-row cap disappears.** On CPU, TabPFN and TabICL are capped at
   5,000 training rows. On a GPU the documented ceiling is a million, so we can
   finally run the full OULAD cohort.
3. **We answer the question the CPU run left open**: gradient boosting is
   supposed to catch up once it has enough data. On CPU we could never give it
   enough. Here we can.

Previous results, all CPU-only, from `tabfm/RESULTS.md` in the repo:

| | gradient boosting | TabICL |
|---|---:|---:|
| PR-AUC @ 4,000 rows | 0.7348 | **0.7585** |
| predict 4,000 students | 0.017 s | **93 s** |

TabICL won 9 runs out of 9 — but only up to 4,000 rows, and ~5,000x slower.
Both of those limits are CPU limits. Let's remove them.

---

## Before you run anything — two Kaggle settings

Open the **⋮** menu (top right) → **Accelerator** and **Internet**:

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **GPU T4 x2** (or P100) | Without this `torch.cuda.is_available()` is False and nothing below works |
| **Internet** | **On** | Needed to install the packages and download model weights |

**Internet requires phone verification on Kaggle.** If the toggle is greyed
out, that is why — verify your account first (Settings → Phone Verification).
It is a one-time thing and the usual reason this notebook fails at cell 1.

Free GPU quota is about **30 hours a week**. This notebook uses well under one.


## 1. Install, and get the data

Kaggle already has torch, pandas, scikit-learn. We add the two foundation
models and clone the project repo, which carries the OULAD data as parquet —
no separate upload needed.

In [ ]:
!pip install -q tabpfn tabicl 2>&1 | tail -3
print("installed")

In [ ]:
import os, subprocess, sys, shutil

REPO   = "/kaggle/working/OULAD-Model"
BRANCH = "dev"          # where the work lands; main can lag behind it

def sh(*args, check=True):
    """Run a command and FAIL LOUDLY. Swallowing a git error here produces a
    stale checkout and a baffling AttributeError several cells later."""
    r = subprocess.run(args, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {' '.join(args)}\n{out}")
    return out

# Always re-clone rather than updating in place.
#
# Updating a shallow clone does not work and fails quietly: `git clone --depth 1`
# implies --single-branch, so remote.origin.fetch covers only the branch it
# cloned. Fetching any other branch lands in FETCH_HEAD without ever creating
# origin/<branch>, and `reset --hard origin/<branch>` then dies with "Needed a
# single revision". /kaggle/working is scratch and a shallow clone takes about
# two seconds, so starting clean is both simpler and correct.
shutil.rmtree(REPO, ignore_errors=True)
sh("git", "clone", "--depth", "1", "--branch", BRANCH,
   "https://github.com/ndg13b/OULAD-Model.git", REPO)
print("HEAD:", sh("git", "-C", REPO, "log", "-1", "--format=%h %s"))

for path in (f"{REPO}/src", f"{REPO}/tabfm"):
    if path not in sys.path:
        sys.path.insert(0, path)

# Python caches modules: an earlier `import tabfm_bench` in this kernel would be
# reused even with fresh files on disk. Drop them so the next import reads disk.
for mod in [m for m in list(sys.modules)
            if m == "tabfm_bench" or m == "oulad" or m.startswith("oulad.")]:
    del sys.modules[mod]

import tabpfn, tabicl
print("tabpfn", tabpfn.__version__, "| tabicl", getattr(tabicl, "__version__", "2.x"))

from oulad import load, features
print("OULAD package imported")

In [ ]:
import tabfm_bench as T

# Self-check: catch a stale checkout HERE, not three sections later.
assert hasattr(T, "prepare_oulad"), (
    "tabfm_bench is stale — it has no prepare_oulad. Re-run the cell above, "
    "then Run > Restart & clear cell outputs and run from the top."
)

print("CPU row caps the package enforces (irrelevant now — we have a GPU):")
for k, v in T.CPU_LIMITS.items():
    print(f"   {k:12s} {v:>7,}")

# Build the feature table ONCE, here, where you can watch it happen.
#
# Reading 10.6M clickstream rows and building 45 features takes a few seconds.
# Section 5 asks for the same cutoff nine times; without this it would redo
# that work every iteration, silently, which looks exactly like a hang.
print()
frame = T.prepare_oulad(cutoff_day=28)

**Why this is first.** `pip install` can change the installed PyTorch. If that
happens *after* you have already imported torch, the copy in memory is stale and
`torch.cuda.is_available()` can report something that is no longer true.
Installing before anything else means every cell below runs against the final
versions.

If a later cell complains that torch and CUDA disagree, use
**Run → Restart & clear cell outputs** and run from the top. That fixes almost
every version-mismatch symptom.

**If the clone failed**, Internet is off — see the settings note above. Nothing
below will work until it is on.

## 2. What hardware did we get?

`nvidia-smi` is the command for asking an NVIDIA GPU about itself. It is the
first thing to run whenever a GPU is involved, and the first thing to run when
something has gone wrong.

In [ ]:
!nvidia-smi

Read that table. The parts that matter:

- **Name** — `Tesla T4` or `Tesla P100`. Kaggle's free tiers.
- **Memory-Usage** — e.g. `0MiB / 15360MiB`. This is **VRAM**: memory *on the
  card*, completely separate from the machine's normal RAM. When you hit
  `CUDA out of memory`, this is what filled up, and no amount of system RAM
  helps.
- **CUDA Version** — the driver's CUDA. It must be ≥ the version PyTorch was
  built against. On Kaggle this is pre-matched, which is why nobody has install
  trouble here and everybody has it on their own machine.
- **Processes** — what is using the card. Useful when memory is mysteriously
  full: something you ran earlier still holds it.

In [ ]:
import torch, platform

print("python          ", platform.python_version())
print("torch           ", torch.__version__)
print("CUDA available  ", torch.cuda.is_available())

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("device          ", p.name)
    print("VRAM            ", f"{p.total_memory / 1024**3:.1f} GiB")
    print("CUDA cores (SMs)", p.multi_processor_count)
    DEVICE = "cuda"
else:
    print()
    print("!! No GPU. Set Accelerator to GPU in the ⋮ menu and rerun.")
    print("   Everything below still runs, just on CPU and much slower.")
    DEVICE = "cpu"

`torch.cuda.is_available()` is the only check you ever really need. If it is
`False`, nothing else about CUDA matters yet.

### What CUDA is, in two sentences

CUDA is NVIDIA's system for running ordinary computation — not graphics — on a
graphics card. **You will not write any.** You write `device="cuda"` and PyTorch,
which was built on top of CUDA, does the rest.

### Why a GPU helps *this* kind of work

A CPU has a few very fast, very flexible cores. A GPU has thousands of slow,
simple ones. If the work is "do the same arithmetic to a million numbers at
once" the GPU wins enormously; if it is branchy sequential logic the CPU wins.

A transformer — which is what TabPFN and TabICL are — is mostly large matrix
multiplications. That is exactly the first kind. Let's measure it.

## 3. Measuring the difference — and the mistake everyone makes

Multiply two 4000x4000 matrices on each device and time it.

But there is a trap, and it is worth meeting on purpose.

In [ ]:
import time
import torch

N = 4000
a_cpu, b_cpu = torch.randn(N, N), torch.randn(N, N)

t0 = time.perf_counter()
_ = a_cpu @ b_cpu
cpu_time = time.perf_counter() - t0
print(f"CPU: {cpu_time:.3f} s")

In [ ]:
if DEVICE == "cuda":
    a, b = a_cpu.to("cuda"), b_cpu.to("cuda")
    torch.cuda.synchronize()          # make sure the copy finished

    # --- The WRONG way ---
    t0 = time.perf_counter()
    _ = a @ b
    wrong = time.perf_counter() - t0

    # --- The RIGHT way ---
    t0 = time.perf_counter()
    _ = a @ b
    torch.cuda.synchronize()
    right = time.perf_counter() - t0

    print(f"GPU, timed naively : {wrong*1000:8.2f} ms   <- not real")
    print(f"GPU, timed properly: {right*1000:8.2f} ms")
    print(f"CPU                : {cpu_time*1000:8.2f} ms")
    print()
    print(f"actual speedup: {cpu_time/right:.0f}x")

**The trap.** GPU operations are *asynchronous*. When Python reaches `a @ b`,
it hands the job to the GPU and immediately moves on — it does not wait. So the
naive timing measures how long it took to *ask*, not how long the work took.
That is why the first number is absurdly small.

`torch.cuda.synchronize()` blocks until the GPU has actually finished. **Any
GPU timing without it is meaningless.** This is the single most common
beginner's error, and it produces results that look like a miracle.

The honest speedup is typically 10–50x for this shape of work.

### The other cost: getting data onto the card

The GPU cannot see your normal RAM. Data has to be copied across, and the copy
is not free.

In [ ]:
if DEVICE == "cuda":
    x = torch.randn(4000, 4000)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    x_gpu = x.to("cuda")
    torch.cuda.synchronize()
    print(f"copying 64 MB to the GPU: {(time.perf_counter()-t0)*1000:.1f} ms")
    print()
    print(f"VRAM in use now: {torch.cuda.memory_allocated()/1024**2:.0f} MiB")
    print(f"VRAM total     : {torch.cuda.get_device_properties(0).total_memory/1024**2:.0f} MiB")

    del a, b, x_gpu, a_cpu, b_cpu
    torch.cuda.empty_cache()
    print(f"after cleanup  : {torch.cuda.memory_allocated()/1024**2:.0f} MiB")

Two practical rules fall out of this:

1. **Move data once, not in a loop.** A transfer inside a loop can cost more
   than the computation it feeds.
2. **`del` plus `torch.cuda.empty_cache()` is how you free VRAM.** If you hit
   `CUDA out of memory`, this — or restarting the kernel — is the fix. Watch
   the numbers above actually drop.

That is genuinely all the CUDA you need to use these models.

## 4. Does the GPU change the picture?

Same comparison as the CPU run, same 4,000-row split, so the numbers are
directly comparable. Only the hardware changed.

Reference, from the CPU run: TabICL 0.7585 PR-AUC, 93 seconds to predict.

In [ ]:
import time
data = T.sample_oulad(n_train=4000, n_test=4000, seed=0)
print(data.describe())

results = [T.run_majority(data), T.run_logistic(data), T.run_gbm(data)]

t0 = time.perf_counter()
results.append(T.run_tabicl(data, device=DEVICE))
print(f"\nTabICL total wall-clock: {time.perf_counter()-t0:.1f} s")

T.print_report(results, data)

Compare the `predict s` column against **93 seconds** on CPU.

The accuracy should be essentially unchanged — same model, same data, same
answer. Only the time moved. That is the whole value of the GPU here: it does
not make the model better, it makes it *usable*.

## 5. The question the CPU could not answer

Here is the open one. The claim is that foundation models win on small data and
gradient boosting catches up given enough. On CPU the sweep stopped at 4,000
rows with TabICL still ahead by +0.024 — so we never saw the crossover, if
there is one.

Now we can go to the full cohort. Test set stays fixed at 4,000 throughout, so
every row of this table is scored on exactly the same students; only the
training context grows.

Expect a few minutes.

In [ ]:
import numpy as np, pandas as pd, time

# The grouped split holds out 30% for testing, so ~19,250 training rows exist.
SIZES = [250, 500, 1000, 2000, 4000, 8000, 12000, 16000, 19000]

rows = []
for n in SIZES:
    d = T.sample_oulad(n_train=n, n_test=4000, seed=0)
    actual = len(d.X_train)

    gbm = T.run_gbm(d)
    t0 = time.perf_counter()
    tic = T.run_tabicl(d, device=DEVICE)
    wall = time.perf_counter() - t0

    rows.append({
        "n_train": actual,
        "base_rate": round(d.base_rate, 4),
        "logistic": round(T.run_logistic(d).pr_auc, 4),
        "gbm": round(gbm.pr_auc, 4),
        "tabicl": round(tic.pr_auc, 4) if np.isfinite(tic.pr_auc) else np.nan,
        "gap": (round(tic.pr_auc - gbm.pr_auc, 4)
                if np.isfinite(tic.pr_auc) else np.nan),
        "tabicl_s": round(wall, 1),
    })
    print(f"  n={actual:>6,}  gbm {rows[-1]['gbm']:.4f}  "
          f"tabicl {rows[-1]['tabicl']}  gap {rows[-1]['gap']:+}  ({wall:.0f}s)")

    if actual < n:   # asked for more than the split can supply
        print(f"  ({actual:,} is the whole training pool — stopping here)")
        break

sweep = pd.DataFrame(rows)
sweep

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

ax1.plot(sweep.n_train, sweep.gbm, marker="o", label="gradient boosting",
         color="#2a78d6")
ax1.plot(sweep.n_train, sweep.tabicl, marker="o", label="TabICL",
         color="#eb6834")
ax1.set_xscale("log")
ax1.set_xlabel("training rows (log scale)")
ax1.set_ylabel("PR-AUC")
ax1.set_title("Does gradient boosting catch up?")
ax1.legend()
ax1.grid(alpha=.3)

ax2.axhline(0, color="#52514e", lw=1)
ax2.plot(sweep.n_train, sweep.gap, marker="o", color="#1baf7a")
ax2.set_xscale("log")
ax2.set_xlabel("training rows (log scale)")
ax2.set_ylabel("TabICL minus gradient boosting")
ax2.set_title("The gap — crossover is where this hits zero")
ax2.grid(alpha=.3)

plt.tight_layout(); plt.show()

last = sweep.iloc[-1]
print(f"At {last.n_train:,.0f} training rows the gap is {last.gap:+.4f}")
print("Positive = TabICL still ahead. Negative = gradient boosting has overtaken.")

**Read the right-hand panel.** The line crossing zero is the crossover — the
amount of data at which gradient boosting overtakes. If it never crosses, that
is a genuinely interesting finding on this dataset, and the opposite of what
the conventional wisdom predicts.

Two cautions before reading too much into it:

- **This is one split.** A gap of ±0.01 is noise. Section 7 reruns the ranking
  across seeds, which is what makes it trustworthy.
- **Gradient boosting is untuned here.** It is on defaults, which is a fair
  comparison against an untuned TabICL but *not* the strongest the incumbent
  can be. A tuned gradient boosting would move that line.

## 6. The regime the claim is actually about

OULAD is big, messy and weakly separable. The published claim for these models
is about *small* tables — a few hundred to a few thousand rows — which is where
most real scientific datasets live.

These are from OpenML, the benchmark suite the TabPFN papers evaluate on.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

DATASETS = [
    ("credit-g", 31),                            # 1,000 x 20, credit risk
    ("diabetes", 37),                            # 768 x 8
    ("blood-transfusion-service-center", 1464),  # 748 x 4
    ("qsar-biodeg", 1494),                       # 1,055 x 41
]

def tiny_compare(name, did):
    try:
        b = fetch_openml(data_id=did, as_frame=True, parser="auto")
    except Exception as e:
        print(f"  {name:36s} could not fetch: {type(e).__name__}")
        return None

    X = b.data
    y = pd.Series(b.target).astype("category").cat.codes.to_numpy()
    if len(np.unique(y)) != 2:
        print(f"  {name:36s} skipped (not binary)")
        return None

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.4, random_state=0, stratify=y)
    cat_idx = [X.columns.get_loc(c) for c in X.columns
               if str(X[c].dtype) in ("object", "category", "str")]

    d = T.Dataset(X_train=Xtr, X_test=Xte, y_train=ytr, y_test=yte,
                  categorical_indices=cat_idx)
    gbm = T.run_gbm(d)
    tic = T.run_tabicl(d, device=DEVICE)
    return {
        "dataset": name, "rows": len(X), "features": X.shape[1],
        "base_rate": round(d.base_rate, 3),
        "gbm": round(gbm.pr_auc, 4),
        "tabicl": round(tic.pr_auc, 4) if np.isfinite(tic.pr_auc) else np.nan,
        "gap": (round(tic.pr_auc - gbm.pr_auc, 4)
                if np.isfinite(tic.pr_auc) else np.nan),
    }

small = [r for r in (tiny_compare(n, i) for n, i in DATASETS) if r]
small_df = pd.DataFrame(small)
small_df

If the `gap` column is mostly positive and clearly larger than the gaps in
section 5, that is the claim reproducing: the advantage is biggest where data
is scarcest.

If it is not, that is worth knowing too — and more informative than a benchmark
table, because you ran it.

## 7. Is any of this real? (seeds)

One split proves nothing. The CPU run's strongest evidence was not a margin but
a *count*: TabICL won 9 out of 9 across three seeds. The sign being consistent
matters more than any single number.

In [ ]:
seed_rows = []
for seed in [0, 1, 2]:
    for n in [1000, 8000]:
        d = T.sample_oulad(n_train=n, n_test=4000, seed=seed)
        gbm = T.run_gbm(d)
        tic = T.run_tabicl(d, device=DEVICE)
        seed_rows.append({
            "n_train": len(d.X_train), "seed": seed,
            "gbm": round(gbm.pr_auc, 4),
            "tabicl": round(tic.pr_auc, 4) if np.isfinite(tic.pr_auc) else np.nan,
            "gap": (round(tic.pr_auc - gbm.pr_auc, 4)
                    if np.isfinite(tic.pr_auc) else np.nan),
        })
        print(f"  n={n:>6,} seed={seed}  gap {seed_rows[-1]['gap']:+}")

seeds_df = pd.DataFrame(seed_rows)
print()
print(seeds_df.groupby("n_train")["gap"].agg(["mean", "min", "max", "count"]).round(4))
wins = (seeds_df.gap > 0).sum()
print(f"\nTabICL ahead in {wins} of {len(seeds_df)} runs")

A gap whose **sign** is stable across seeds is a result. A gap that changes
sign is noise, however large any single value looks.

## 8. TabPFN (optional — needs a token)

TabPFN will not download its weights without a Prior Labs licence token. This
is **not** a Hugging Face gate and `HF_TOKEN` will not help — it is a licence
check that runs locally before any download.

To enable it:

1. Register at <https://ux.priorlabs.ai> and accept the licence on the
   **Licenses** tab
2. Copy your API key from <https://ux.priorlabs.ai/account>
3. In Kaggle: **Add-ons → Secrets → Add secret**, name it `TABPFN_TOKEN`
4. Rerun the cell below

Skip it if you would rather not — everything above stands without it. Note the
weights for the current models are **non-commercial** licensed.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["TABPFN_TOKEN"] = UserSecretsClient().get_secret("TABPFN_TOKEN")
    print("token loaded from Kaggle Secrets")
except Exception as e:
    print(f"no TABPFN_TOKEN ({type(e).__name__}) — skipping TabPFN")

d = T.sample_oulad(n_train=4000, n_test=4000, seed=0)
res = T.run_tabpfn(d, device=DEVICE)
if np.isfinite(res.pr_auc):
    print(f"\nTabPFN PR-AUC {res.pr_auc:.4f}  "
          f"(fit {res.fit_seconds:.1f}s, predict {res.predict_seconds:.1f}s)")
    T.print_report([T.run_gbm(d), T.run_tabicl(d, device=DEVICE), res], d)
else:
    print(f"\nTabPFN unavailable: {res.note}")

## 9. What you now know that you did not before

**About GPUs.** You read `nvidia-smi`. You know VRAM is not RAM, that transfers
cost time, and that GPU timing without `torch.cuda.synchronize()` is fiction.
You never wrote a line of CUDA, because for this work nobody does.

**About these models.** You have the crossover curve from section 5 — on your
own data, at sizes a laptop cannot reach. Whatever it shows is a better answer
to "should I bet on these?" than any benchmark table, because it is your data
and your baseline.

**Honest limits of what is above:**

- Gradient boosting ran on defaults throughout. A tuned incumbent is the fairer
  fight and would move the lines.
- Section 5 is a single split. Section 7 checks the ranking but not the
  magnitude; a publishable interval wants repeated grouped CV.
- Everything is one cutoff (day 28) on one cohort. The OULAD project's real
  question — how early you can usefully predict — is a different sweep.

**Things worth trying from here:**

```python
T.sample_oulad(n_train=8000, n_test=4000, cutoff_day=7)    # a much harder task
T.run_gbm(d, max_iter=500, learning_rate=0.05)             # give gbm a real fight
T.run_tabicl(d, device=DEVICE, n_estimators=8)             # more ensemble members
```

To keep results, commit them back to the repo as `tabfm/RESULTS-GPU.md`
alongside the CPU run, with the GPU name and package versions — the CPU file
is the template.
